# PathLens-GNN — Kaggle registered run
The cells are ordered to preserve the sealed test: baseline suite, tuning, three-seed confirmation, model freeze, one final evaluation, then artifact export.

In [ ]:
import os
import pathlib
import subprocess

GIT_REF = "planning"  # Switch to main or a release tag after the planning PR is merged.
REPO = pathlib.Path("/kaggle/working/PathLens-GNN")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            GIT_REF,
            "--single-branch",
            "https://github.com/aryonmt/PathLens-GNN.git",
            str(REPO),
        ],
        check=True,
    )
os.chdir(REPO)
subprocess.run(["python", "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)

In [ ]:
import gzip
import shutil
import urllib.request

raw_dir = pathlib.Path("/kaggle/working/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
source = raw_dir / "biosnap.tsv"
if not source.exists():
    archive = raw_dir / "biosnap.tsv.gz"
    urllib.request.urlretrieve(
        "https://snap.stanford.edu/biodata/datasets/10002/files/ChG-Miner_miner-chem-gene.tsv.gz",
        archive,
    )
    with gzip.open(archive, "rb") as inp, source.open("wb") as out:
        shutil.copyfileobj(inp, out)
subprocess.run(
    ["pathlens", "--source", str(source), "--output", "/kaggle/working/data/processed"],
    check=True,
)

In [ ]:
subprocess.run(
    [
        "python",
        "scripts/run_registered_experiments.py",
        "--processed",
        "/kaggle/working/data/processed",
        "--output",
        "/kaggle/working/output/registered",
    ],
    check=True,
)

In [ ]:
subprocess.run(
    [
        "python",
        "scripts/run_tuning.py",
        "--processed",
        "/kaggle/working/data/processed",
        "--output",
        "/kaggle/working/output/tuning",
    ],
    check=True,
)

In [ ]:
import json

leaders = json.loads(pathlib.Path("/kaggle/working/output/tuning/leaderboard.json").read_text())
leaders[:2]

In [ ]:
subprocess.run(
    [
        "python",
        "scripts/run_confirmation.py",
        "--processed",
        "/kaggle/working/data/processed",
        "--leaderboard",
        "/kaggle/working/output/tuning/leaderboard.json",
        "--output",
        "/kaggle/working/output/confirmation",
    ],
    check=True,
)
confirmation = json.loads(
    pathlib.Path("/kaggle/working/output/confirmation/confirmation.json").read_text()
)
confirmation["selected"]

In [ ]:
chosen = confirmation["selected"]["seed_13_checkpoint"]
freeze_record = "/kaggle/working/output/model-freeze.json"
subprocess.run(
    [
        "python",
        "scripts/freeze_model.py",
        "--checkpoint",
        chosen,
        "--output",
        freeze_record,
        "--decision",
        "Best preregistered validation result after the fixed compute budget",
    ],
    check=True,
)
subprocess.run(
    [
        "python",
        "scripts/evaluate_checkpoint.py",
        "--processed",
        "/kaggle/working/data/processed",
        "--checkpoint",
        chosen,
        "--freeze-record",
        freeze_record,
        "--output",
        "/kaggle/working/output/final-evaluation.json",
        "--confirm-sealed-test",
    ],
    check=True,
)
subprocess.run(
    [
        "python",
        "scripts/export_artifact.py",
        "--processed",
        "/kaggle/working/data/processed",
        "--checkpoint",
        chosen,
        "--output",
        "/kaggle/working/output/artifact/pathlens-v1",
        "--model-version",
        "pathlens-v1-candidate",
    ],
    check=True,
)
shutil.make_archive(
    "/kaggle/working/pathlens-v1-candidate",
    "zip",
    "/kaggle/working/output/artifact/pathlens-v1",
)